# Minimal GRPO vs. SFT in PyTorch on GSM8K Math Problems

This notebook implements **Group Relative Policy Optimization (GRPO)** (from *DeepSeekMath* / *DeepSeek-R1*) from scratch in pure PyTorch and compares it head-to-head against a **Supervised Fine-Tuning (SFT)** baseline on GSM8K-style math word problems.

### Key Highlights for Google Colab GPUs (T4 16GB):
1. **Critic-Free RL**: GRPO eliminates the PPO value/critic network by sampling $G$ completions per prompt and normalizing rewards within each group:
   $$A_{i} = \frac{r_i - \text{mean}(r_1, \dots, r_G)}{\text{std}(r_1, \dots, r_G) + \epsilon}$$
2. **Zero-Extra-VRAM Reference Policy**: By training a LoRA adapter on the policy $\pi_\theta$, we compute reference log-probabilities $\log \pi_{\text{ref}}$ simply by toggling `with model.disable_adapter():`—saving a full copy of the LLM in GPU memory.
3. **Rule-Based Verifiable Rewards**: Combines a **format reward** (`<think>...</think>\n<answer>...</answer>`) and an **exact numerical accuracy reward**.

In [ ]:
# 1. Install dependencies on Google Colab
!pip install -q torch transformers datasets peft accelerate tqdm matplotlib
!pip install -U "torchao>=0.16.0"

In [8]:
!git clone https://github.com/ehamid/RL-GRPO.git
%cd RL-GRPO

Cloning into 'RL-GRPO'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 26 (delta 5), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 47.91 KiB | 7.98 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/RL-GRPO/RL-GRPO


In [ ]:
# 2. Verify GPU and run the full Base vs. SFT vs. GRPO experiment
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

# Run the end-to-end comparison script (takes ~6-8 minutes on a Colab T4 GPU)
!python train_and_compare.py \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --train-samples 200 \
    --eval-samples 50 \
    --sft-steps 60 \
    --grpo-steps 60 \
    --group-size 4

CUDA available: True
Device: Tesla T4
W1109 04:50:06.244000 5636 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1109 04:50:06.278000 5636 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[setup] Using device=cuda, dtype=torch.bfloat16, model=Qwen/Qwen2.5-0.5B-Instruct
[data] Loaded 200 train examples and 50 test examples.
Loading weights: 100% 290/290 [00:00<00:00, 4626.96it/s]

[1/3] Evaluating Base Model...
                                             
[2/3] Training SFT Baseline (LoRA)...
SFT Training:  78% 47/60 [01:10<00:21,  1.67s/it, loss=0.4037]

In [10]:
# 3. Display the generated comparison plots and sample completions
import json
from IPython.display import Image, display

display(Image(filename='outputs/comparison_results.png'))

with open('outputs/comparison_metrics.json') as f:
    results = json.load(f)

for model_name in ['Base', 'SFT', 'GRPO']:
    sample = results['eval_summary'][model_name]['samples'][0]
    print(f'=== {model_name} Sample Generation ===')
    print('Question:', sample['question'])
    print('Ground Truth:', sample['ground_truth'], '| Predicted:', sample['predicted'])
    print('Completion:\n', sample['completion'])
    print()

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/comparison_results.png'